In [2]:

import numpy as np
import pandas as pd
from scipy.stats import f, ncf
from scipy.optimize import brentq

# =========================================================
# POWER ANALYSIS: REVISED SUPPLY CHAIN RESILIENCE MODEL
# =========================================================

# Study settings
N_actual = 405
alpha = 0.05
target_power = 0.90
f2_assumed = 0.15

# RCC predictors:
# 1. RI
# 2. DIQ
# 3. SA
# 4. RI x DIQ
# 5. DIQ x SA
# 6. RI x SA
total_predictors = 6


def regression_power(N, f2, total_p, tested_p, alpha=0.05):
    """
    Power for a fixed-model multiple regression R²-increase test.
    Uses the noncentral F distribution.
    """
    df_num = tested_p
    df_den = N - total_p - 1

    if df_den <= 0 or f2 < 0:
        return np.nan

    # Reduced model excludes the tested predictors
    noncentrality = f2 * (N - total_p + tested_p)

    critical_f = f.ppf(
        1 - alpha, df_num, df_den
    )

    power = ncf.sf(
        critical_f, df_num, df_den, noncentrality
    )

    return power


def minimum_sample_size(f2, total_p, tested_p,
                        target_power=0.90, alpha=0.05):
    """Find the smallest N achieving the target power."""
    for N in range(total_p + 5, 100000):
        power = regression_power(
            N, f2, total_p, tested_p, alpha
        )
        if power >= target_power:
            return N, power
    raise ValueError("Required sample size not found.")


def minimum_detectable_f2(N, total_p, tested_p,
                          target_power=0.90, alpha=0.05):
    """Find the minimum f² detectable at a given sample size."""
    if N <= total_p + 1:
        raise ValueError("Sample size is too small.")

    def objective(f2):
        return regression_power(
            N, f2, total_p, tested_p, alpha
        ) - target_power

    return brentq(objective, 1e-8, 10)


# ---------------------------------------------------------
# A. A PRIORI POWER: MEDIUM EFFECT SIZE f² = 0.15
# ---------------------------------------------------------

# Joint test of all three interaction terms
N_joint, achieved_joint_power = minimum_sample_size(
    f2=f2_assumed,
    total_p=total_predictors,
    tested_p=3,
    target_power=target_power,
    alpha=alpha
)

# Individual test of one interaction, controlling for
# the other main effects and interaction terms
N_individual, achieved_individual_power = minimum_sample_size(
    f2=f2_assumed,
    total_p=total_predictors,
    tested_p=1,
    target_power=target_power,
    alpha=alpha
)


# ---------------------------------------------------------
# B. SENSITIVITY: ACTUAL SAMPLE SIZE N = 405
# ---------------------------------------------------------

f2_joint = minimum_detectable_f2(
    N=N_actual,
    total_p=total_predictors,
    tested_p=3,
    target_power=target_power,
    alpha=alpha
)

f2_individual = minimum_detectable_f2(
    N=N_actual,
    total_p=total_predictors,
    tested_p=1,
    target_power=target_power,
    alpha=alpha
)


# ---------------------------------------------------------
# C. DISPLAY RESULTS
# ---------------------------------------------------------

results = pd.DataFrame([
    {
        "Analysis": "Joint interaction test",
        "Tested predictors": 3,
        "Effect size f²": f2_assumed,
        "Minimum N": N_joint,
        "Power at minimum N": achieved_joint_power
    },
    {
        "Analysis": "Individual interaction test",
        "Tested predictors": 1,
        "Effect size f²": f2_assumed,
        "Minimum N": N_individual,
        "Power at minimum N": achieved_individual_power
    },
    {
        "Analysis": "Joint sensitivity at N=405",
        "Tested predictors": 3,
        "Effect size f²": f2_joint,
        "Minimum N": N_actual,
        "Power at specified effect": target_power
    },
    {
        "Analysis": "Individual sensitivity at N=405",
        "Tested predictors": 1,
        "Effect size f²": f2_individual,
        "Minimum N": N_actual,
        "Power at specified effect": target_power
    }
])

print("POWER ANALYSIS RESULTS")
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nInterpretation:")
print(f"Actual sample size: {N_actual}")
print(f"Joint detectable effect size f²: {f2_joint:.4f}")
print(f"Individual detectable effect size f²: {f2_individual:.4f}")


POWER ANALYSIS RESULTS
                       Analysis  Tested predictors  Effect size f²  Minimum N  Power at minimum N  Power at specified effect
         Joint interaction test                  3          0.1500        102              0.9017                        NaN
    Individual interaction test                  1          0.1500         78              0.9039                        NaN
     Joint sensitivity at N=405                  3          0.0356        405                 NaN                     0.9000
Individual sensitivity at N=405                  1          0.0264        405                 NaN                     0.9000

Interpretation:
Actual sample size: 405
Joint detectable effect size f²: 0.0356
Individual detectable effect size f²: 0.0264
